# UAV crowd counting: UAV-Dot vs P2PNet

Run both head-point detectors on your own aerial image. Each model marks every detected head as a point; the count is
the number of points.

| VisDrone2020-CC test set (900 frames) | Count MAE | F1 @ 16 px | L-mAP | fp16 latency, RTX 4060 Laptop |
|---|---:|---:|---:|---:|
| **UAV-Dot** (released weights) | 25.0 | 0.726 | 51.0 | **29 ms** |
| **P2PNet** (trained in this project) | **21.9** | 0.725 | **52.5** | 115 ms |

**Use a GPU runtime** (*Runtime → Change runtime type → T4 GPU*) to see realistic speed; a CPU runtime works but is slower.

- **P2PNet** downloads from Hugging Face: [`uwaisasghar/p2pnet-visdrone-head-counting`](https://huggingface.co/uwaisasghar/p2pnet-visdrone-head-counting).
- **UAV-Dot** uses the released weights from the [authors' mirror](https://github.com/up-count/uav-dot), checked by SHA-256 and not re-hosted.

Code, evaluation protocol and limitations: [github.com/owaisasghar/uav-crowd-counting](https://github.com/owaisasghar/uav-crowd-counting).

In [ ]:
# Get the code and the one extra package (Colab already has PyTorch, Pillow and huggingface_hub)
!git clone -q --depth 1 https://github.com/owaisasghar/uav-crowd-counting.git
!pip -q install segmentation-models-pytorch==0.5.0

In [ ]:
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from huggingface_hub import snapshot_download
from PIL import Image

REPO = Path("uav-crowd-counting")
sys.path.insert(0, str(REPO / "huggingface_space"))
import uavdot_runtime

# P2PNet: inference package from Hugging Face
p2pnet_dir = Path(snapshot_download("uwaisasghar/p2pnet-visdrone-head-counting"))
spec = importlib.util.spec_from_file_location("p2pnet_runtime", p2pnet_dir / "p2pnet_runtime.py")
p2pnet_runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p2pnet_runtime)
p2pnet, device = p2pnet_runtime.load_model(p2pnet_dir / "model.safetensors", device="auto")

# UAV-Dot: released weights from the authors' mirror (about 330 MB, SHA-256 checked)
uavdot_checkpoint = uavdot_runtime.download_checkpoint(Path.home() / ".cache" / "uavdot")
uavdot, _ = uavdot_runtime.load_model(uavdot_checkpoint, device=device)
print("Both models loaded on", device)

## Choose an aerial image

Upload one when prompted, or set `IMAGE_PATH` to a file path or an image URL. The models were trained on drone footage
from roughly 20–80 m looking down; other viewpoints work less well.

In [ ]:
import urllib.request

IMAGE_PATH = None  # e.g. "my_drone_frame.jpg" or "https://.../frame.jpg"

if IMAGE_PATH is None:
    from google.colab import files  # Colab's upload widget
    IMAGE_PATH = next(iter(files.upload()))
elif str(IMAGE_PATH).startswith("http"):
    IMAGE_PATH, _ = urllib.request.urlretrieve(IMAGE_PATH, "input_image")
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"{IMAGE_PATH}: {image.width}x{image.height}")

In [ ]:
use_half = device.type == "cuda"

def timed(predict, *args, **kwargs):
    predict(*args, **kwargs)  # warm-up run, not timed
    return predict(*args, **kwargs)

uav = timed(uavdot_runtime.predict_points, uavdot, image, device, threshold=0.2, use_half=use_half)
p2p = timed(p2pnet_runtime.predict_points, p2pnet, image, device, threshold=0.5, max_side=1920, use_half=use_half)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
for ax, name, result, picture in [
    (axes[0], "UAV-Dot", uav, uavdot_runtime.annotate_image(image, uav["points"], 0.2)),
    (axes[1], "P2PNet", p2p, p2pnet_runtime.annotate_image(image, p2p["points"], 0.5)),
]:
    ax.imshow(picture)
    ax.set_title(f"{name}: {result['count']} heads, {result['inference_ms']:.0f} ms on {device.type}", fontsize=15)
    ax.axis("off")
plt.tight_layout()
plt.show()

Timings cover model inference after a warm-up run. Each model uses its evaluated threshold (UAV-Dot 0.2, P2PNet 0.5)
and full resolution. The test-set results and known weaknesses (night footage, very low altitude) are in the
[report](https://github.com/owaisasghar/uav-crowd-counting/blob/main/REPORT.md).

## Optional: the interactive app

This launches the same side-by-side Gradio app as the repository's `huggingface_space/`, with a temporary public link.

In [ ]:
!pip -q install "gradio>=6,<7"
%cd uav-crowd-counting/huggingface_space
import app
app.demo.queue().launch(share=True)